# SIGMOD Exp 4: Concurrent Transactions

Closed-loop concurrent transaction benchmark over the maintained Q14 build-side state. Each reader worker executes many independent read transactions; writer workers execute update transactions concurrently. This notebook wraps `concurrent_tx_bench` and produces three figures:

1. Read-transaction throughput vs reader threads
2. Fixed-point average read latency at 4R/1W
3. Update-volume sweep of read blocking

In [ ]:
import sys, subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', 'pandas', 'matplotlib', 'numpy'])
print('done')

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path('../../').resolve()
sys.path.append(str(ROOT / 'benches'))

from sigmod_exp_common import TOL, apply_paper_style, ensure_dirs, run_checked, resolve_tpch_file, resolve_update_file

apply_paper_style(ROOT)

EXP_DIR = (ROOT / 'benches' / 'sigmod_exp4_concurrency').resolve()
DATA_DIR = EXP_DIR / 'data'
FIGS_DIR = EXP_DIR / 'figs'
ensure_dirs(DATA_DIR, FIGS_DIR)

TPCH_DIR = (ROOT / 'benches' / 'sigmod' / 'tpch_data').resolve()
GEN_UPDATES = (ROOT / 'benches' / 'sigmod' / 'generate_updates.py').resolve()
BIN = ROOT / 'target' / 'release' / 'concurrent_tx_bench'

SF = '1.0'
BUCKET_NUM = 2048
WARMUP = 1
REPEAT = 5
TRIM = 1
READ_TX_SIZE = 2048
UPDATE_TX_SIZE = 256
READ_ROUNDS = 4
FIXED_UPDATE_PCT = 1.0
READER_THREADS = [1, 2, 4, 8]
SWEEP_PCTS = [0.01, 0.2, 0.4, 0.6, 0.8, 1.0, 2.0, 5.0]
SERIES = [
    ('snap', 'nr'),
    ('ivmh', 'nr'),
    ('heap', 'wr'),
    ('chain', 'wr'),
    ('par', 'wr'),
]
STYLE = {
    ('snap', 'nr'): ('SNAP', TOL['red'], ':', 'x'),
    ('ivmh', 'nr'): ('IVMH', TOL['yellow'], '--', 'P'),
    ('heap', 'wr'): ('MONO-WR', TOL['blue'], '-', 'o'),
    ('chain', 'wr'): ('DUAL-WR', TOL['cyan'], '-', 's'),
    ('par', 'wr'): ('EPOCH-WR', TOL['green'], '-', 'D'),
}

PART_FILE = resolve_tpch_file(TPCH_DIR, 'part', SF)
LINEITEM_FILE = resolve_tpch_file(TPCH_DIR, 'lineitem_probe', SF, '_1995-09-01_1995-10-01.tbl')

print('PART    :', PART_FILE)
print('LINEITEM:', LINEITEM_FILE)
print('BIN     :', BIN)

In [ ]:
print('Building concurrent_tx_bench...')
run_checked(['cargo', 'build', '--release', '--bin', 'concurrent_tx_bench'], ROOT)
print('Build OK')

In [ ]:
def ensure_update_file(pct):
    try:
        return resolve_update_file(TPCH_DIR, SF, pct, 'uniform')
    except FileNotFoundError:
        print(f'Generating update file for {pct}%...')
        run_checked([sys.executable, str(GEN_UPDATES), str(PART_FILE), str(pct), str(SF), '--output-dir', str(TPCH_DIR)], ROOT)
        return resolve_update_file(TPCH_DIR, SF, pct, 'uniform')


def run_point(table_type, repair_mode, reader_threads, writer_threads, update_pct, output_csv):
    updates_file = ensure_update_file(update_pct)
    result = run_checked([
        str(BIN),
        '--part-file', str(PART_FILE),
        '--lineitem-file', str(LINEITEM_FILE),
        '--updates-file', str(updates_file),
        '--table-type', table_type,
        '--repair-mode', repair_mode,
        '--bucket-num', str(BUCKET_NUM),
        '--reader-threads', str(reader_threads),
        '--writer-threads', str(writer_threads),
        '--read-tx-size', str(READ_TX_SIZE),
        '--update-tx-size', str(UPDATE_TX_SIZE),
        '--read-rounds', str(READ_ROUNDS),
        '--warmup', str(WARMUP),
        '--repeat', str(REPEAT),
        '--trim', str(TRIM),
        '--update-pct', str(update_pct),
        '--output-csv', str(output_csv),
    ], ROOT, quiet=True)
    for line in result.stderr.splitlines():
        if '[iter' in line or 'Average' in line or 'avg_read' in line or 'reader_tx_tput' in line:
            print(' ', line)

SCALABILITY_CSV = DATA_DIR / 'sigmod_exp4_scalability.csv'
if SCALABILITY_CSV.exists():
    SCALABILITY_CSV.unlink()
for reader_threads in READER_THREADS:
    for table_type, repair_mode in SERIES:
        print(f'scalability: readers={reader_threads} {table_type}/{repair_mode}')
        run_point(table_type, repair_mode, reader_threads, 1, FIXED_UPDATE_PCT, SCALABILITY_CSV)

df_scale = pd.read_csv(SCALABILITY_CSV)
display(df_scale[['table_type', 'repair_mode', 'reader_threads', 'total_ms', 'avg_read_latency_ms', 'avg_read_wait_ms', 'reader_tx_throughput']].head())

In [ ]:
FIXED_CSV = DATA_DIR / 'sigmod_exp4_fixed.csv'
if FIXED_CSV.exists():
    FIXED_CSV.unlink()
for table_type, repair_mode in SERIES:
    print(f'fixed-point: {table_type}/{repair_mode}')
    run_point(table_type, repair_mode, 4, 1, FIXED_UPDATE_PCT, FIXED_CSV)

df_fixed = pd.read_csv(FIXED_CSV)
display(df_fixed[['table_type', 'repair_mode', 'avg_read_latency_ms', 'p95_read_latency_ms', 'avg_read_wait_ms', 'reader_tx_throughput']])

In [ ]:
SWEEP_CSV = DATA_DIR / 'sigmod_exp4_update_sweep.csv'
if SWEEP_CSV.exists():
    SWEEP_CSV.unlink()
for pct in SWEEP_PCTS:
    for table_type, repair_mode in SERIES:
        print(f'update sweep: pct={pct} {table_type}/{repair_mode}')
        run_point(table_type, repair_mode, 4, 1, pct, SWEEP_CSV)

df_sweep = pd.read_csv(SWEEP_CSV)
display(df_sweep[['table_type', 'repair_mode', 'update_pct', 'avg_read_latency_ms', 'avg_read_wait_ms']].head())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14.0, 4.3))

for key, (label, color, linestyle, marker) in STYLE.items():
    table_type, repair_mode = key
    sub = df_scale[(df_scale['table_type'] == table_type) & (df_scale['repair_mode'] == repair_mode)].sort_values('reader_threads')
    axes[0].plot(sub['reader_threads'], sub['reader_tx_throughput'], color=color, linestyle=linestyle, marker=marker, linewidth=1.8, markersize=5, label=label)
axes[0].set_title('Read-Tx Scalability')
axes[0].set_xlabel('Reader Threads')
axes[0].set_ylabel('Read Tx Throughput (tx/s)')
axes[0].grid(True, linestyle='--', linewidth=0.6, alpha=0.6)

labels = []
values = []
colors = []
for key, (label, color, _, _) in STYLE.items():
    table_type, repair_mode = key
    row = df_fixed[(df_fixed['table_type'] == table_type) & (df_fixed['repair_mode'] == repair_mode)].iloc[0]
    labels.append(label)
    values.append(float(row['avg_read_latency_ms']))
    colors.append(color)
axes[1].bar(labels, values, color=colors, edgecolor='black', linewidth=0.4)
axes[1].set_title('Fixed 4R/1W Read Latency')
axes[1].set_ylabel('Avg Read Tx Latency (ms)')
axes[1].tick_params(axis='x', rotation=25)
axes[1].grid(True, axis='y', linestyle='--', linewidth=0.6, alpha=0.6)

for key, (label, color, linestyle, marker) in STYLE.items():
    table_type, repair_mode = key
    sub = df_sweep[(df_sweep['table_type'] == table_type) & (df_sweep['repair_mode'] == repair_mode)].sort_values('update_pct')
    axes[2].plot(sub['update_pct'], sub['avg_read_wait_ms'], color=color, linestyle=linestyle, marker=marker, linewidth=1.8, markersize=5, label=label)
axes[2].set_title('Update-Volume Sweep')
axes[2].set_xlabel('Update Volume (%)')
axes[2].set_ylabel('Avg Read Wait (ms)')
axes[2].grid(True, linestyle='--', linewidth=0.6, alpha=0.6)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', ncol=5, bbox_to_anchor=(0.5, 1.08), framealpha=0.95)
fig.tight_layout()
out_pdf = FIGS_DIR / 'sigmod_exp4_concurrency.pdf'
fig.savefig(out_pdf, format='pdf')
plt.show()
print('Saved', out_pdf)